In [1]:
import numpy as np
import pandas as pd
from scipy import stats

inc = pd.read_csv("data/processed/incidents_geo.csv",
                  dtype={"station": str, "nearest_station": str, "neighbourhood_id": str})
ctx = pd.read_csv("data/processed/neighbourhood_context.csv", dtype={"neighbourhood_id": str})

d = inc[inc["neighbourhood_id"].notna() & inc["dist_nearest_km"].notna()].copy()
d["log_wait"] = np.log(d["wait_s"])
d["year_c"] = d["year"].astype(str)
print(len(d))

26092


/var/folders/98/p89mc3nd1d34f9jndy3v2lz00000gn/T/ipykernel_27783/3263199069.py:5: DtypeWarning: Columns (5: Incident_Station_Area) have mixed types. Specify dtype option on import or set low_memory=False.
  inc = pd.read_csv("data/processed/incidents_geo.csv",


In [2]:
def fit(df, numeric, categorical=(), y="log_wait", cluster="neighbourhood_id"):
    X = df[list(numeric)].astype(float)
    if categorical:
        X = pd.concat([X, pd.get_dummies(df[list(categorical)], drop_first=True).astype(float)], axis=1)
    X.insert(0, "const", 1.0)
    Xv, yv = X.to_numpy(), df[y].to_numpy()
    n, k = Xv.shape
    bread = np.linalg.inv(Xv.T @ Xv)
    beta = bread @ Xv.T @ yv
    resid = yv - Xv @ beta
    codes, uniq = pd.factorize(df[cluster])
    G = len(uniq)
    scores = np.zeros((G, k))
    np.add.at(scores, codes, Xv * resid[:, None])
    cov = (G / (G - 1)) * ((n - 1) / (n - k)) * bread @ (scores.T @ scores) @ bread
    se = np.sqrt(np.diag(cov))
    p = 2 * stats.t.sf(np.abs(beta / se), G - 1)
    crit = stats.t.ppf(0.975, G - 1)
    out = pd.DataFrame({"coef": beta, "se": se, "p": p}, index=X.columns)
    out["pct_effect"] = (np.exp(out["coef"]) - 1) * 100
    out["ci_lo_pct"] = (np.exp(out["coef"] - crit * out["se"]) - 1) * 100
    out["ci_hi_pct"] = (np.exp(out["coef"] + crit * out["se"]) - 1) * 100
    return out

rows = []
def show(label, df, dist="dist_nearest_km"):
    m = fit(df, [dist, "is_weekend"], ["daypart", "year_c"])
    r, night = m.loc[dist], m.loc["daypart_night"]
    rows.append({"check": label, "n": len(df), "per_km_pct": r.pct_effect, "lo": r.ci_lo_pct,
                 "hi": r.ci_hi_pct, "night_pct": night.pct_effect})
    print(f"{label:32s} n={len(df):6d}  per km {r.pct_effect:5.1f}% [{r.ci_lo_pct:.1f}, {r.ci_hi_pct:.1f}]   night {night.pct_effect:5.1f}%")

In [3]:
show("baseline (nearest station)", d)

baseline (nearest station)       n= 26092  per km  29.5% [26.9, 32.1]   night  16.1%


In [4]:
da = d[d["dist_assigned_km"].notna()]
show("nearest, same rows", da)
show("assigned station", da, "dist_assigned_km")
show("assigned is also nearest", da[da["station"] == da["nearest_station"]])
show("assigned, within 5 km", da[da["dist_assigned_km"] <= 5], "dist_assigned_km")

nearest, same rows               n= 26066  per km  29.5% [26.9, 32.1]   night  16.2%
assigned station                 n= 26066  per km  17.9% [14.0, 21.8]   night  16.3%
assigned is also nearest         n= 21439  per km  30.7% [28.0, 33.4]   night  17.5%
assigned, within 5 km            n= 25893  per km  23.9% [22.0, 25.9]   night  16.8%


In [5]:
for lo, hi in [(10, 900), (60, 3600), (120, 3600), (120, 900)]:
    show(f"waits {lo} to {hi} s", d[d["wait_s"].between(lo, hi)])

waits 10 to 900 s                n= 26041  per km  29.2% [26.7, 31.8]   night  16.1%
waits 60 to 3600 s               n= 25990  per km  29.1% [26.5, 31.7]   night  15.3%
waits 120 to 3600 s              n= 25592  per km  28.9% [26.3, 31.6]   night  13.6%
waits 120 to 900 s               n= 25541  per km  28.6% [26.1, 31.2]   night  13.6%


In [6]:
effects = [fit(d[d["year"] != y], ["dist_nearest_km", "is_weekend"], ["daypart", "year_c"]).loc["dist_nearest_km", "pct_effect"]
           for y in sorted(d["year"].unique())]
print("distance effect when each year is left out:", round(min(effects), 1), "to", round(max(effects), 1))

distance effect when each year is left out: 29.2 to 30.2


In [7]:
m_s = fit(d, ["dist_nearest_km", "is_weekend"], ["daypart", "year_c"], y="wait_s")
print("seconds per extra km:", round(m_s.loc["dist_nearest_km", "coef"], 1), "+/-", round(1.96 * m_s.loc["dist_nearest_km", "se"], 1))
print("extra seconds at night:", round(m_s.loc["daypart_night", "coef"], 1))

seconds per extra km: 80.8 +/- 8.3
extra seconds at night: 42.3


In [8]:
d["band"] = pd.qcut(d["dist_nearest_km"], 4, labels=["b1", "b2", "b3", "b4"]).astype(str)
print(d.groupby("band")["dist_nearest_km"].agg(["min", "max"]).round(2))
mb = fit(d, ["is_weekend"], ["band", "daypart", "year_c"])
mb.loc[["band_b2", "band_b3", "band_b4"], ["pct_effect", "ci_lo_pct", "ci_hi_pct"]].round(1)

       min   max
band            
b1    0.03  0.61
b2    0.61  0.95
b3    0.96  1.39
b4    1.39  5.77


,pct_effect,ci_lo_pct,ci_hi_pct
band_b2,14.0,12.0,16.1
band_b3,28.6,25.7,31.6
band_b4,43.2,38.9,47.7


In [9]:
d["dist2"] = d["dist_nearest_km"] ** 2
mq = fit(d, ["dist_nearest_km", "dist2", "is_weekend"], ["daypart", "year_c"])
mq.loc[["dist_nearest_km", "dist2"], ["coef", "p"]].round(3)

,coef,p
dist_nearest_km,0.379,0.000
dist2,-0.049,0.005


In [10]:
pts = inc[inc["has_coords"]].groupby(["Latitude", "Longitude"]).size()
print("incidents with coordinates:", int(pts.sum()), "| distinct locations:", len(pts))
print("share of incidents at a location shared with another incident:", round(pts[pts >= 2].sum() / pts.sum(), 3))
print("most incidents at one point:", pts.max())

unplaced = inc[inc["has_coords"] & inc["neighbourhood_id"].isna()]
placed = inc[inc["neighbourhood_id"].notna()]
print("unplaced:", len(unplaced), "| median wait", unplaced["wait_s"].median(), "vs placed", placed["wait_s"].median())

incidents with coordinates: 26166 | distinct locations: 14627
share of incidents at a location shared with another incident: 0.637
most incidents at one point: 47
unplaced: 74 | median wait 375.0 vs placed 303.0


In [11]:
def residuals(df, numeric, categorical=(), y="log_wait"):
    X = df[list(numeric)].astype(float)
    if categorical:
        X = pd.concat([X, pd.get_dummies(df[list(categorical)], drop_first=True).astype(float)], axis=1)
    X.insert(0, "const", 1.0)
    beta = np.linalg.lstsq(X.to_numpy(), df[y].to_numpy(), rcond=None)[0]
    return df[y].to_numpy() - X.to_numpy() @ beta

def neighbourhood_delay(df, rcol="resid"):
    g = df.groupby("neighbourhood_id")[rcol].agg(n="size", mean_resid="mean").reset_index()
    N, G = len(df), len(g)
    within = ((df[rcol] - df.groupby("neighbourhood_id")[rcol].transform("mean")) ** 2).sum() / (N - G)
    tau2 = max(0.0, g["mean_resid"].var() - (within / g["n"]).mean())
    g["shrink"] = tau2 / (tau2 + within / g["n"])
    g["shrunk"] = g["shrink"] * g["mean_resid"]
    sd = np.sqrt(g["shrink"] * within / g["n"])
    g["adj_shrunk_pct"] = (np.exp(g["shrunk"]) - 1) * 100
    g["lo_pct"] = (np.exp(g["shrunk"] - 1.96 * sd) - 1) * 100
    g["hi_pct"] = (np.exp(g["shrunk"] + 1.96 * sd) - 1) * 100
    return g

adj = pd.read_csv("data/processed/neighbourhood_adjusted_delay.csv", dtype={"neighbourhood_id": str})
five = adj.loc[adj["clear_slow"] & adj["top_risk"], "neighbourhood_id"].tolist()

d["resid"] = residuals(d, ["dist_nearest_km", "is_weekend"], ["daypart", "year_c"])
g0 = neighbourhood_delay(d)
base_top10 = set(g0.nlargest(10, "adj_shrunk_pct")["neighbourhood_id"])

def variant(label, df, numeric, categorical):
    df = df.copy()
    df["resid"] = residuals(df, numeric, categorical)
    g = neighbourhood_delay(df)
    both = g0.merge(g, on="neighbourhood_id", suffixes=("_0", "_v"))
    rho = stats.spearmanr(both["adj_shrunk_pct_0"], both["adj_shrunk_pct_v"])[0]
    top10 = set(g.nlargest(10, "adj_shrunk_pct")["neighbourhood_id"])
    still = (g[g["neighbourhood_id"].isin(five)]["lo_pct"] > 0).sum()
    print(f"{label:28s} rank corr {rho:.2f} | top-10 overlap {len(top10 & base_top10)} | clearly slow {(g['lo_pct'] > 0).sum()} | the five still clearly slow {still}/5")

time_cat = ["daypart", "year_c"]
variant("baseline", d, ["dist_nearest_km", "is_weekend"], time_cat)
variant("assigned-station distance", da, ["dist_assigned_km", "is_weekend"], time_cat)
variant("waits of 2 min or more", d[d["wait_s"] >= 120], ["dist_nearest_km", "is_weekend"], time_cat)
variant("years 2011 to 2022", d[d["year"] <= 2022], ["dist_nearest_km", "is_weekend"], time_cat)
variant("flexible distance (bands)", d, ["is_weekend"], ["band"] + time_cat)
variant("with distance squared", d, ["dist_nearest_km", "dist2", "is_weekend"], time_cat)

baseline                     rank corr 1.00 | top-10 overlap 10 | clearly slow 30 | the five still clearly slow 5/5
assigned-station distance    rank corr 0.91 | top-10 overlap 6 | clearly slow 36 | the five still clearly slow 5/5
waits of 2 min or more       rank corr 0.97 | top-10 overlap 9 | clearly slow 41 | the five still clearly slow 4/5
years 2011 to 2022           rank corr 0.98 | top-10 overlap 10 | clearly slow 39 | the five still clearly slow 4/5
flexible distance (bands)    rank corr 0.97 | top-10 overlap 8 | clearly slow 32 | the five still clearly slow 5/5
with distance squared        rank corr 0.99 | top-10 overlap 9 | clearly slow 30 | the five still clearly slow 5/5


In [12]:
codes, uniq = pd.factorize(d["neighbourhood_id"])
n_per = np.bincount(codes)
sd_resid = d["resid"].std()

def spread_and_flags(res):
    means = np.bincount(codes, weights=res) / n_per
    z = means / (sd_resid / np.sqrt(n_per))
    return means.std(), int((np.abs(z) > 1.96).sum())

real_spread, real_flags = spread_and_flags(d["resid"].to_numpy())
rng = np.random.default_rng(42)
sims = np.array([spread_and_flags(rng.permutation(d["resid"].to_numpy())) for _ in range(1000)])

print("real spread between neighbourhoods:", round(real_spread, 3), "| flagged as different:", real_flags)
print("shuffled: spread averages", sims[:, 0].mean().round(3), "(max", sims[:, 0].max().round(3), ") | flagged averages", sims[:, 1].mean().round(1), "(max", sims[:, 1].max(), ")")

real spread between neighbourhoods: 0.074 | flagged as different: 73
shuffled: spread averages 0.029 (max 0.035 ) | flagged averages 7.9 (max 19.0 )


In [13]:
pd.DataFrame(rows).round(2).to_csv("data/processed/robustness_distance.csv", index=False)
print("saved", len(rows), "checks")

saved 9 checks


## What I learned from the robustness checks

I wanted to know whether my main findings depend on the choices I made along the way, so I changed those choices one at a time. The distance effect stayed clearly positive in all nine versions. Trimming extreme waits moved it only between 28.6% and 29.2% per km. Leaving out any single year kept it between 29.2% and 30.2%. Modelling seconds instead of percentages gave about 81 extra seconds per km, plus about 42 seconds at night.

The one choice that matters is how I measure distance. With the nearest station, each extra km is associated with about 29.5% longer waits. With the station listed in the incident file, it drops to about 17.9%. That listed station is sometimes very far away (up to 38.8 km), which may be a recording error or help from a distant station. The file doesn't say which truck arrived first. Where the listed station is also the nearest, the effect is 30.7%. So I'll report about 18% to 31% per km, with the nearest-station figure as my main estimate, and I won't quote a single number on its own.

The shape of the distance effect is close to a straight line. Compared with the closest quarter of incidents, the next three quarters wait about 14%, 29% and 43% longer, with a slight flattening at greater distances. The night effect was consistent too, at roughly 14–17% in every version.

Locations are approximate. About 64% of incidents share an exact coordinate with another incident, and one point has 47. Distances are therefore reliable only to a few hundred metres. The 74 incidents that fall outside every neighbourhood outline are slower (375 s median versus 303 s), but there are too few to change the picture.

The neighbourhood findings held up. The ranking correlated 0.91 to 0.99 with the baseline across six alternative setups, and between 6 and 10 of the top 10 stayed the same. The five "slow and high-risk" neighbourhoods stayed clearly slower than expected in four of the six setups, and four of the five in the other two. The shuffle test showed the differences are not chance: the real spread between neighbourhoods was 0.074, against 0.029 on average (0.035 at most) when I shuffled the data, and 73 neighbourhoods were flagged against about 8 by chance.

What this does not show is why a neighbourhood is slower than expected. It only shows that the differences are real and not caused by the distance measure, the trimming or the years I chose.